


* `Few shot learning`: Le Few-shot learning désigne un type d'apprentissage où un modèle est capable de comprendre des tâches avec très peu d'exemples d'apprentissage. Il s'oppose à l'apprentissage traditionnel qui nécessite une grande quantité de données étiquetées
<!--
prompt_imputation = """
Voici quelques exemples de données où certaines valeurs sont manquantes :

Exemple 1 : 
Transaction_Amount : 1000, Time_of_Transaction : 15, Device_Used : Mobile, Location : Paris, Payment_Method : Debit Card
Exemple 2 : 
Transaction_Amount : 500, Time_of_Transaction : 18, Device_Used : Desktop, Location : New York, Payment_Method : Credit Card

Maintenant, remplis les valeurs manquantes pour la ligne suivante :
Transaction_Amount : NA, Time_of_Transaction : 20, Device_Used : Mobile, Location : NA, Payment_Method : Debit Card
"""
-->
* `Chain-of-Thought (CoT)`: Chain-of-Thought (CoT) est une technique d'apprentissage qui consiste à faire réfléchir un modèle étape par étape avant de donner une réponse. Au lieu de fournir une réponse immédiate, le modèle décompose le problème en étapes intermédiaires qui aident à arriver à une solution plus robuste et précise.
<!--
prompt_imputation_cot = """
Réfléchis étape par étape pour remplir la valeur manquante dans cette ligne de données. Voici les informations disponibles :

Transaction_Amount : NA, Time_of_Transaction : 14, Device_Used : Mobile, Location : NA, Payment_Method : Debit Card

Étape 1 : Quelle est la valeur probable pour Transaction_Amount en fonction du type de transaction et des autres informations ?
Étape 2 : Quelle localisation pourrait correspondre à cette transaction en fonction de l'appareil utilisé et de l'heure de la transaction ?
Étape 3 : Basé sur ces informations, remplis les valeurs manquantes de manière cohérente.
"""
-->

* `LangChain et LangGraph`: LangChain est une bibliothèque qui permet de créer des chaînes de tâches (pipelines) complexes en utilisant des modèles de langage. Elle est particulièrement utile pour automatiser des processus qui nécessitent des modèles de langage puissants, comme GPT-3 ou GPT-4, en enchaînant différentes étapes de traitement du texte. LangGraph est un concept qui est utilisé pour créer des graphes de tâches. Cela permet d'organiser des tâches complexes en différents nœuds et de créer des dépendances entre ces nœuds. Dans un LangGraph, chaque nœud pourrait être une tâche comme imputation, génération de données synthétiques, ou toute autre étape de traitement.
<!--
from langchain.chains import SequentialChain
from langchain.prompts import PromptTemplate

# Étape 1 : Créer des templates de prompts pour chaque tâche
imputation_prompt = PromptTemplate(input_variables=["data"], template="Remplis les valeurs manquantes dans cette ligne : {data}")
synthetic_data_prompt = PromptTemplate(input_variables=["data"], template="Génère 10 nouveaux exemples de transactions frauduleuses similaires à cette ligne : {data}")

# Étape 2 : Créer une chaîne séquentielle pour orchestrer les étapes
chain = SequentialChain(
    chains=[imputation_chain, synthetic_data_chain],
    input_variables=["data"],
    output_variables=["imputed_data", "synthetic_data"]
)

# Exécuter la chaîne sur des données spécifiques
result = chain.run(data=your_data_here)
print(result)
-->
* `FAISS / Qdrant`: FAISS (Facebook AI Similarity Search) et Qdrant sont des systèmes permettant de faire de la recherche par similarité dans de grands ensembles de données. Ces outils sont utilisés pour effectuer des recherches efficaces dans des bases de données contenant des embeddings ou des représentations vectorielles d'objets (comme des textes, des images, etc.).  FAISS : Utilisé principalement pour la recherche par similarité dans des embeddings de grande dimension. FAISS peut être utilisé pour rechercher rapidement des exemples similaires dans des grands ensembles de données.  Qdrant : C'est une alternative à FAISS, utilisée pour la gestion et la recherche de vecteurs dans des bases de données plus dynamiques.
<!--
import faiss
import numpy as np

# Création d'un index FAISS pour les embeddings des transactions réelles
transactions = np.array([your_transaction_embeddings])  # Embeddings des transactions existantes
index = faiss.IndexFlatL2(transactions.shape[1])  # Créer un index pour la recherche par similarité
index.add(transactions)  # Ajouter les transactions réelles à l'index

# Générer un exemple synthétique et obtenir ses embeddings
synthetic_data_embedding = np.array([generate_synthetic_data_embedding()])

# Rechercher des transactions similaires dans les données existantes
D, I = index.search(synthetic_data_embedding, k=5)  # Trouver les 5 plus proches voisins

print("Indices des transactions similaires:", I)
-->

In [114]:
!pip install -U accelerate
!pip install transformers datasets
!pip install fsspec==2026.1.0
!pip install -U transformers kernels torch
!pip install -U sentencepiece protobuf
!pip install -U ipywidgets jupyter
!jupyter nbextension enable --py widgetsnbextension
!pip install -U transformers accelerate sentencepiece protobuf huggingface_hub



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
Using cached fsspec-2025.10.0-py3-none-any.whl (200 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.1.0
    Uninstalling fsspec-2026.1.0:
      Successfully uninstalled fsspec-2026.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
s3fs 2026.1.0 requires fsspec==2026.1.0, but you have fsspec 2025.10.0 which is incompatible.
langchain-huggingface 1.2.0 requires huggingface-hub<1.0.0,>=0.33.4, but you have huggingface-hub 1.3.7 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
  Using cached fsspec-2026.1.0-py3-none-any.whl.metadata (10 kB)
Using cached fsspec-2026.1.0-py3-none-

In [115]:
import pandas as pd

data = pd.read_csv(
    "s3://riva/Fraud Detection Dataset.csv",
    storage_options={"anon": False}
)
data.head()


,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Time_of_Transaction,Device_Used,Location,Previous_Fraudulent_Transactions,Account_Age,Number_of_Transactions_Last_24H,Payment_Method,Fraudulent
0,T1,4174,1292.76,ATM Withdrawal,16.0,Tablet,San Francisco,0,119,13,Debit Card,0
1,T2,4507,1554.58,ATM Withdrawal,13.0,Mobile,New York,4,79,3,Credit Card,0
2,T3,1860,2395.02,ATM Withdrawal,NaN,Mobile,NaN,3,115,9,NaN,0
3,T4,2294,100.10,Bill Payment,15.0,Desktop,Chicago,4,3,4,UPI,0
4,T5,2130,1490.50,POS Payment,19.0,Mobile,San Francisco,2,57,7,Credit Card,0


In [116]:
prompt_fewshot = """
Tu es un expert en détection de fraude bancaire.

TA TÂCHE : génère 50 transactions synthétiques au format JSON (liste de dicts).

SCHÉMA EXACT :
- TransactionID : "T" + 9 chiffres
- UserID : entier 1000-999999
- TransactionAmount : float 1-10000€ (fraude souvent >1000€)
- TransactionType : ["Online Purchase","POS Payment","ATM Withdrawal","Bill Payment","Bank Transfer"]
- TimeofTransaction : entier 0-23 (fraude souvent 22h-4h)
- DeviceUsed : ["Desktop","Mobile","Tablet","Unknown Device"]
- Location : ["Los Angeles","New York","Seattle","Miami","Chicago","Houston","Boston","San Francisco"]
- PreviousFraudulentTransactions : entier 0-10 (fraude >2)
- AccountAge : entier 0-500 (fraude souvent compte jeune <100j)
- NumberofTransactionsLast24H : entier 0-50 (fraude >10)
- PaymentMethod : ["UPI","Credit Card","Net Banking","Debit Card","Invalid Method"]
- Fraudulent : 0 ou 1 (10% doivent être 1)

EXEMPLES (few-shot) :

EXEMPLE FRAUDE :
{"TransactionID":"T111864396","UserID":4406,"TransactionAmount":406.81,"TransactionType":"Online Purchase",
 "TimeofTransaction":17,"DeviceUsed":"Desktop","Location":"Los Angeles","PreviousFraudulentTransactions":0,"AccountAge":22,"NumberofTransactionsLast24H":1,"PaymentMethod":"Invalid Method","Fraudulent":1}

EXEMPLE NON-FRAUDE :
{"TransactionID":"T111872663","UserID":2133,"TransactionAmount":33.33,"TransactionType":"POS Payment",
 "TimeofTransaction":16,"DeviceUsed":"Tablet","Location":"New York","PreviousFraudulentTransactions":1,"AccountAge":95,"NumberofTransactionsLast24H":12,"PaymentMethod":"Net Banking","Fraudulent":0}

RAISONNEMENT (chain-of-thought) :
Pour chaque transaction :
1. Décide si fraude (10% oui) : montant élevé + Invalid Method + beaucoup txns + compte jeune → FRAUDE
2. Choisis valeurs cohérentes selon décision
3. Formate JSON strictement

RÉPONDS UNIQUEMENT avec la liste JSON des 50 transactions, rien d'autre.
"""


In [ ]:
import requests
import pandas as pd
import json

HF_TOKEN = "VOTRE TOKEN"
API_URL = "https://api-inference.huggingface.co/models/meta-llama/Llama-3.2-3B-Instruct"
headers = {"Authorization": f"Bearer {HF_TOKEN}"}

frauds = data[data['Fraudulent'] == 1].head(5)

prompt = f"""Génère 10 nouvelles transactions frauduleuses au format JSON array.

Exemples: {frauds.to_dict('records')}

Génère 10 nouvelles transactions différentes avec les mêmes champs en JSON array:"""

response = requests.post(
    API_URL,
    headers=headers,
    json={
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": 1000,
            "temperature": 0.8,
            "return_full_text": False
        }
    }
)

result = response.json()
print("Réponse:", result)

try:
    generated_text = result[0]['generated_text']
    start = generated_text.find('[')
    end = generated_text.rfind(']') + 1
    json_data = json.loads(generated_text[start:end])
    
    synthetic_df = pd.DataFrame(json_data)
    synthetic_df['Fraudulent'] = 1
    
    print(f"\n✅ {len(synthetic_df)} transactions")
    print(synthetic_df)
    synthetic_df.to_csv("synthetic_fraud_data.csv", index=False)
except:
    print("Erreur extraction")

Réponse: {'error': 'https://api-inference.huggingface.co is no longer supported. Please use https://router.huggingface.co instead.'}
Erreur extraction


In [ ]:
import os
from huggingface_hub import login
from transformers import pipeline
os.environ["HF_TOKEN"] = "VOTRE TOKEN"
# 1) token
login(token=os.environ["HF_TOKEN"])   

# 2) Modèles
GPT_MODEL   = "EleutherAI/gpt-neo-2.7B"                 # GPT 
LLAMA_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"      # LLaMA

# 3) Pipelines
gpt_gen = pipeline("text-generation", model=GPT_MODEL, token=os.environ["HF_TOKEN"], device_map="auto")
llama_gen = pipeline("text-generation", model=LLAMA_MODEL, token=os.environ["HF_TOKEN"], device_map="auto")

# 4) Prompt few-shot (format CSV)
prompt = """Génère 5 transactions frauduleuses au format CSV (une ligne = une transaction)
Colonnes exactes:
Transaction_Amount,Time_of_Transaction,Device_Used,Location,Payment_Method,Fraudulent

Exemples fraude:
2450,2,Mobile,Lagos,Credit Card,1
1800,23,Desktop,New York,Debit Card,1

Maintenant génère 5 nouvelles lignes CSV.
Réponds uniquement par les 5 lignes (sans commentaire).
"""

print(" GPT ")
out_gpt = gpt_gen(prompt, max_new_tokens=200, do_sample=True, temperature=0.8, top_p=0.9)
print(out_gpt[0]["generated_text"])

print("LLaMA (TinyLlama) ")
out_llama = llama_gen(prompt, max_new_tokens=200, do_sample=True, temperature=0.8, top_p=0.9)
print(out_llama[0]["generated_text"])


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading weights:   0%|          | 0/420 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-2.7B
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0...31}.attn.attention.masked_bias | UNEXPECTED |  | 
transformer.h.{0...30}.attn.attention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 GPT 


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Génère 5 transactions frauduleuses au format CSV (une ligne = une transaction)
Colonnes exactes:
Transaction_Amount,Time_of_Transaction,Device_Used,Location,Payment_Method,Fraudulent

Exemples fraude:
2450,2,Mobile,Lagos,Credit Card,1
1800,23,Desktop,New York,Debit Card,1

Maintenant génère 5 nouvelles lignes CSV.
Réponds uniquement par les 5 lignes (sans commentaire).

A:

En général, en base à la procédure de réponse, un string de réponse qui contient le message d'erreur, peut être transformé en n-éléments. Par exemple:

If the result is null, it means that the input string is empty.

Pour chaque élément, le caractère de la ligne en bas de la fichier est la valeur de la ligne en bas, et le caractère de la ligne à la gauche est la valeur de la ligne à la droite.

A:

Les choses sont assez simples. 

Une ligne est un élément de l'en-tête du fichier CSV.
Une ligne est un élément de l'en-t
LLaMA (TinyLlama) 
Génère 5 transactions frauduleuses au format CSV (une ligne = une transaction)
C

In [ ]:
import pandas as pd
from io import StringIO

def csv_lines_to_df(text):
    lines = [l.strip() for l in text.splitlines() if l.count(",") >= 5]
    csv_text = "\n".join(lines)
    df = pd.read_csv(StringIO(csv_text), header=None)
    df.columns = ["Transaction_Amount","Time_of_Transaction","Device_Used","Location","Payment_Method","Fraudulent"]
    return df

df_gpt = csv_lines_to_df(out_gpt[0]["generated_text"])
df_llama = csv_lines_to_df(out_llama[0]["generated_text"])

display(df_gpt.head())
display(df_llama.head())


,Transaction_Amount,Time_of_Transaction,Device_Used,Location,Payment_Method,Fraudulent
0,Transaction_Amount,Time_of_Transaction,Device_Used,Location,Payment_Method,Fraudulent
1,2450,2,Mobile,Lagos,Credit Card,1
2,1800,23,Desktop,New York,Debit Card,1


,Transaction_Amount,Time_of_Transaction,Device_Used,Location,Payment_Method,Fraudulent
0,Transaction_Amount,Time_of_Transaction,Device_Used,Location,Payment_Method,Fraudulent
1,2450,2,Mobile,Lagos,Credit Card,1
2,1800,23,Desktop,New York,Debit Card,1


In [ ]:

# 1. Authentification Hugging Face
from huggingface_hub import login
import os
os.environ["HF_TOKEN"] = "VOTRE TOKEN"
login(token=os.environ["HF_TOKEN"])
# 2. Charger LLaMA 
from transformers import pipeline

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

generator = pipeline(
    task="text-generation",
    model=model_id,
    token=os.environ["HF_TOKEN"],
    device_map="auto"
)

# 3. Few-shot + génération
prompt = """
Tu es un expert en détection de fraude bancaire.

Voici des exemples de transactions FRAUDULEUSES :

Exemple 1:
Transaction_Amount: 2450
Time_of_Transaction: 2
Device_Used: Mobile
Location: Lagos
Payment_Method: Credit Card
Fraudulent: 1

Exemple 2:
Transaction_Amount: 1800
Time_of_Transaction: 23
Device_Used: Desktop
Location: New York
Payment_Method: Debit Card
Fraudulent: 1

Maintenant, génère 5 nouvelles transactions frauduleuses réalistes
au même format.
"""

output = generator(
    prompt,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.8,
    top_p=0.9
)

print(output[0]["generated_text"])


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Tu es un expert en détection de fraude bancaire.

Voici des exemples de transactions FRAUDULEUSES :

Exemple 1:
Transaction_Amount: 2450
Time_of_Transaction: 2
Device_Used: Mobile
Location: Lagos
Payment_Method: Credit Card
Fraudulent: 1

Exemple 2:
Transaction_Amount: 1800
Time_of_Transaction: 23
Device_Used: Desktop
Location: New York
Payment_Method: Debit Card
Fraudulent: 1

Maintenant, génère 5 nouvelles transactions frauduleuses réalistes
au même format.

Transaction_Amount: 2450
Time_of_Transaction: 2
Device_Used: Mobile
Location: Lagos
Payment_Method: Credit Card
Fraudulent: 1

Transaction_Amount: 1800
Time_of_Transaction: 23
Device_Used: Desktop
Location: New York
Payment_Method: Debit Card
Fraudulent: 1

Transaction_Amount: 4500
Time_of_Transaction: 3
Device_Used: Mobile
Location: Lagos
Payment_Method: Credit Card
Fraudulent: 0

Transaction_Amount: 3000
Time_of_Transaction: 2
Device_Used: Desktop
Location: New York
Payment_Method: Debit Card
Fraudulent: 1

Transaction_Amount:

In [121]:
prompt = """Génère 5 transactions frauduleuses au format CSV:
Transaction_Amount,Time_of_Transaction,Device_Used,Location,Payment_Method,Fraudulent
Exemples:
2450,2,Mobile,Lagos,Credit Card,1
1800,23,Desktop,New York,Debit Card,1
Maintenant génère 5 nouvelles lignes.
Réponds uniquement par les 5 lignes CSV.
"""

out = generator(prompt, max_new_tokens=200, do_sample=True, temperature=0.8, top_p=0.9)
print(out[0]["generated_text"])


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Génère 5 transactions frauduleuses au format CSV:
Transaction_Amount,Time_of_Transaction,Device_Used,Location,Payment_Method,Fraudulent
Exemples:
2450,2,Mobile,Lagos,Credit Card,1
1800,23,Desktop,New York,Debit Card,1
Maintenant génère 5 nouvelles lignes.
Réponds uniquement par les 5 lignes CSV.
En cas de fraude, les lignes 2 à 5 ne seront pas générées.
```

Ce script génère un nouveau fichier CSV avec 5 transactions frauduleuses au format CSV. Les 5 transactions générées sont séparées par une ligne. Les transactions ne sont pas générées si une fraude a été commise.


Restart


La base

In [ ]:
import pandas as pd

# 1) Charger ton CSV
import pandas as pd

df = pd.read_csv(
    "s3://riva/Fraud Detection Dataset.csv",
    storage_options={"anon": False}
)

data.head()
fraud_df = df[df["Fraudulent"] == 1].reset_index(drop=True)
non_fraud_df = df[df["Fraudulent"] == 0].reset_index(drop=True)
print("Fraudes :", len(fraud_df), " | Non-fraudes :", len(non_fraud_df))
fraud_df.head()

Index(['Transaction_ID', 'User_ID', 'Transaction_Amount', 'Transaction_Type',
       'Time_of_Transaction', 'Device_Used', 'Location',
       'Previous_Fraudulent_Transactions', 'Account_Age',
       'Number_of_Transactions_Last_24H', 'Payment_Method', 'Fraudulent'],
      dtype='object')
Fraudes : 2510  | Non-fraudes : 48490


,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Time_of_Transaction,Device_Used,Location,Previous_Fraudulent_Transactions,Account_Age,Number_of_Transactions_Last_24H,Payment_Method,Fraudulent
0,T7,4772,544.81,Bill Payment,2.0,Tablet,Boston,3,6,9,UPI,1
1,T28,3433,4519.04,Bill Payment,13.0,Tablet,Boston,0,81,10,Credit Card,1
2,T44,3047,4413.05,Bank Transfer,21.0,Desktop,Boston,4,46,11,Credit Card,1
3,T69,3363,3517.65,Bill Payment,0.0,Desktop,Boston,2,29,8,UPI,1
4,T74,4417,1440.79,Online Purchase,15.0,Tablet,Miami,2,16,1,Net Banking,1


Connexion a hugging face

In [ ]:
import os
from huggingface_hub import InferenceClient
HF_TOKEN = os.getenv("VOTRE TOKEN")  

In [ ]:
import os
from huggingface_hub import InferenceClient
HF_TOKEN = "VOTRE TOKEN"

client = InferenceClient(
    model="meta-llama/Meta-Llama-3-8B-Instruct",   
    token=HF_TOKEN,
)
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"  

Transformer une ligne en phrase

In [125]:
def row_to_description(row):
    return (
        f"User_ID={row['User_ID']}, "
        f"Transaction_Amount={row['Transaction_Amount']}, "
        f"Transaction_Type={row['Transaction_Type']}, "
        f"Time_of_Transaction={row['Time_of_Transaction']}, "
        f"Device_Used={row['Device_Used']}, "
        f"Location={row['Location']}, "
        f"Previous_Fraudulent_Transactions={row['Previous_Fraudulent_Transactions']}, "
        f"Account_Age={row['Account_Age']}, "
        f"Number_of_Transactions_Last_24H={row['Number_of_Transactions_Last_24H']}, "
        f"Payment_Method={row['Payment_Method']}, "
        f"Fraudulent=1"
    )


Prompt few shot

In [126]:
import random

def build_prompt(fraud_df, n_examples=10, n_new=100):
    examples = fraud_df.sample(n=min(n_examples, len(fraud_df)), random_state=42)
    examples_text = "\n".join(
        f"- {row_to_description(row)}"
        for _, row in examples.iterrows()
    )

    prompt = f"""
Tu es un modèle qui génère des exemples réalistes de transactions bancaires FRAUDULEUSES pour entraîner un modèle de détection de fraude.

Voici quelques exemples de transactions frauduleuses existantes (format tabulaire sérialisé) :
{examples_text}

Génère {n_new} nouvelles transactions frauduleuses réalistes.
Contraintes IMPORTANTES :
- Utilise EXACTEMENT le même format que les lignes ci‑dessus.
- Une transaction par ligne.
- Ne mets AUCUN texte explicatif ni commentaire, seulement la liste des lignes.
- Utilise les mêmes noms de champs et le même ordre :
  User_ID, Transaction_Amount, Transaction_Type, Time_of_Transaction, Device_Used, Location, Previous_Fraudulent_Transactions, Account_Age, Number_of_Transactions_Last_24H, Payment_Method, Fraudulent.
- Fraudulent doit toujours être égal à 1.
"""
    return prompt.strip()


Appel du modèle et generation

In [ ]:
def generate_synthetic_frauds(client, model_name, prompt):
    completion = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": "Tu génères des données tabulaires de transactions bancaires frauduleuses. Tu respectes strictement le format demandé."
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0.9,   # plus haut → plus de diversité
        max_tokens=2000, 
    )
    return completion.choices[0].message.content

prompt = build_prompt(fraud_df, n_examples=10, n_new=200)
raw_output = generate_synthetic_frauds(client, MODEL_NAME, prompt)
print(raw_output[:1000])


1111, 2563.57, Bill Payment, 6.0, Desktop, Atlanta, 2, 25, 11, UPI, 1
3456, 1452.91, POS Payment, 14.0, Mobile, New York, 4, 73, 8, Credit Card, 1
2789, 3918.55, Online Purchase, 22.0, Tablet, Chicago, 3, 59, 9, UPI, 1
2198, 6171.44, POS Payment, 19.0, Desktop, Los Angeles, 3, 41, 15, Credit Card, 1
4678, 2473.81, Bill Payment, 2.0, Mobile, Boston, 0, 118, 6, UPI, 1
1256, 6129.77, POS Payment, 11.0, Tablet, Miami, 0, 38, 8, Credit Card, 1
2345, 1827.31, Online Purchase, 15.0, Desktop, Houston, 2, 48, 12, UPI, 1
1290, 2365.15, Bill Payment, 20.0, Mobile, New York, 0, 119, 14, Credit Card, 1
6789, 4756.98, POS Payment, 9.0, Tablet, Atlanta, 4, 85, 5, UPI, 1
9012, 2683.9, Bill Payment, 8.0, Mobile, Boston, 0, 50, 11, Credit Card, 1
3452, 1968.92, POS Payment, 12.0, Desktop, Chicago, 0, 63, 7, UPI, 1
1298, 4393.84, Online Purchase, 25.0, Tablet, Houston, 0, 64, 9, Credit Card, 1
2567, 2664.19, POS Payment, 16.0, Mobile, Los Angeles, 4, 36, 10, UPI, 1
8923, 5187.83, Online Purchase, 18.0, D

affichage

In [128]:
import pandas as pd
import io

# raw_output = texte généré par le LLM
#df = pd.read_csv(io.StringIO(raw_output))

#df.head(10)

In [ ]:
import re
import pandas as pd

def parse_generated_frauds(text):
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
            
        line = re.sub(r'^\d+\.\s*', '', line)
        line = re.sub(r'^\d+\)\s*', '', line)
        
        line = re.sub(r'^[-*]\s*', '', line)
        
        pattern = (
            r"User_ID=(?P<User_ID>\d+).*?"  # User_ID=1234
            r"Transaction_Amount=(?P<Transaction_Amount>[\d\.]+).*?"  # Transaction_Amount=1234.56
            r"Transaction_Type=(?P<Transaction_Type>[^,]+).*?"  # Transaction_Type=POS Payment
            r"Time_of_Transaction=(?P<Time_of_Transaction>[\d\.]+).*?"  # Time_of_Transaction=20.0
            r"Device_Used=(?P<Device_Used>[^,]+).*?"  # Device_Used=Mobile
            r"Location=(?P<Location>[^,]+).*?"  # Location=San Francisco
            r"Previous_Fraudulent_Transactions=(?P<Previous_Fraudulent_Transactions>\d+).*?"  # Previous_Fraudulent_Transactions=3
            r"Account_Age=(?P<Account_Age>\d+).*?"  # Account_Age=45
            r"Number_of_Transactions_Last_24H=(?P<Number_of_Transactions_Last_24H>\d+).*?"  # Number_of_Transactions_Last_24H=12
            r"Payment_Method=(?P<Payment_Method>[^,]+).*?"  # Payment_Method=Credit Card
            r"Fraudulent=(?P<Fraudulent>[01])"  # Fraudulent=1
        )

        m = re.search(pattern, line)
        if m:
            rows.append(m.groupdict())

    synth_df = pd.DataFrame(rows)
    if synth_df.empty:
        print("DEBUG: Aucune ligne parsée. Voici les 5 premières lignes nettoyées :")
        for line in text.splitlines()[:5]:
            cleaned = re.sub(r'^\d+\.\s*', '', line.strip())
            print(f"'{cleaned}'")
        return pd.DataFrame()

    # Cast des types
    synth_df["User_ID"] = synth_df["User_ID"].astype(int)
    synth_df["Transaction_Amount"] = synth_df["Transaction_Amount"].astype(float)
    synth_df["Time_of_Transaction"] = synth_df["Time_of_Transaction"].astype(float)
    synth_df["Previous_Fraudulent_Transactions"] = synth_df["Previous_Fraudulent_Transactions"].astype(int)
    synth_df["Account_Age"] = synth_df["Account_Age"].astype(int)
    synth_df["Number_of_Transactions_Last_24H"] = synth_df["Number_of_Transactions_Last_24H"].astype(int)
    synth_df["Fraudulent"] = synth_df["Fraudulent"].astype(int)

    return synth_df


In [130]:
synth_fraud_df = parse_generated_frauds(raw_output)
synth_fraud_df.head(10)


DEBUG: Aucune ligne parsée. Voici les 5 premières lignes nettoyées :
'1111, 2563.57, Bill Payment, 6.0, Desktop, Atlanta, 2, 25, 11, UPI, 1'
'3456, 1452.91, POS Payment, 14.0, Mobile, New York, 4, 73, 8, Credit Card, 1'
'2789, 3918.55, Online Purchase, 22.0, Tablet, Chicago, 3, 59, 9, UPI, 1'
'2198, 6171.44, POS Payment, 19.0, Desktop, Los Angeles, 3, 41, 15, Credit Card, 1'
'4678, 2473.81, Bill Payment, 2.0, Mobile, Boston, 0, 118, 6, UPI, 1'


""


fusion au besoin

In [131]:
augmented_df = pd.concat([df, synth_fraud_df], ignore_index=True)

print("Taille originale :", len(df))
print("Taille après augmentation :", len(augmented_df))
print(augmented_df["Fraudulent"].value_counts())

#augmented_df.to_csv("Fraud-Detection-Dataset-augmented.csv", index=False)


Taille originale : 51000
Taille après augmentation : 51000
Fraudulent
0    48490
1     2510
Name: count, dtype: int64


Chain-of-Thought (CoT) (raisonne étape par étape)

In [132]:
def cot_prompt(fraud_df, n_new=50):
    example_reasoning = """
RAISONNEMENT ÉTAPE PAR ÉTAPE :
1. Choisir User_ID : nombre aléatoire 4 chiffres (1000-9999)
2. Transaction_Amount : montant suspect 500-5000$ (souvent élevé pour fraudes)
3. Transaction_Type : POS Payment, Online Purchase ou Bill Payment (fréquents fraudes)
4. Time_of_Transaction : heure tardive 20h-6h (suspect)
5. Device_Used : Mobile/Tablet (souvent compromis)
6. Location : ville US différente du domicile habituel
7. Previous_Fraudulent_Transactions : 2-4 (pattern frauduleux)
8. Account_Age : jeune compte <100 jours
9. Number_of_Transactions_Last_24H : élevé 10-15
10. Payment_Method : Credit Card/UPI
11. Fraudulent=1

EXEMPLE appliqué : User_ID=2191, Transaction_Amount=2957.56, ...
"""
    
    return f"""
CHAIN-OF-THOUGHT : Raisonnes étape par étape pour créer des fraudes réalistes.

{example_reasoning}

Maintenant applique ce raisonnement pour générer {n_new} transactions frauduleuses.
POUR CHAQUE transaction :
1. Raisonne → 2. Génère la ligne au format exact
NE SORTS QUE les lignes finales (pas le raisonnement), une par ligne.
"""

prompt_cot = cot_prompt(fraud_df)
raw_cot = generate_synthetic_frauds(client, MODEL_NAME, prompt_cot)
print("\n=== CHAIN-OF-THOUGHT RESULT ===")
print(raw_cot)


=== CHAIN-OF-THOUGHT RESULT ===
Voici les 50 transactions frauduleuses générées conformément au format demandé :

1. User_ID=2621, Transaction_Amount=1842.33, Transaction_Type=Online Purchase, Time_of_Transaction=23:14, Device_Used=Mobile/Tablet, Location=Nashville, Previous_Fraudulent_Transactions=2, Account_Age=25, Number_of_Transactions_Last_24H=13, Payment_Method=Credit Card, Fraudulent=1

2. User_ID=8139, Transaction_Amount=4758.22, Transaction_Type=Bill Payment, Time_of_Transaction=21:50, Device_Used=Tablet, Location=Raleigh, Previous_Fraudulent_Transactions=3, Account_Age=18, Number_of_Transactions_Last_24H=11, Payment_Method=UPI, Fraudulent=1

3. User_ID=2191, Transaction_Amount=2957.56, Transaction_Type=POS Payment, Time_of_Transaction=03:59, Device_Used=Mobile/Tablet, Location=Charleston, Previous_Fraudulent_Transactions=4, Account_Age=85, Number_of_Transactions_Last_24H=10, Payment_Method=Credit Card, Fraudulent=1

4. User_ID=1146, Transaction_Amount=2198.44, Transaction_Ty

In [133]:
cot_df = parse_generated_frauds(raw_cot)
print(cot_df.head())


   User_ID  Transaction_Amount Transaction_Type  Time_of_Transaction  \
0     2621             1842.33  Online Purchase                 23.0   
1     8139             4758.22     Bill Payment                 21.0   
2     2191             2957.56      POS Payment                  3.0   
3     1146             2198.44  Online Purchase                 22.0   
4     8491             3785.69     Bill Payment                  2.0   

     Device_Used    Location  Previous_Fraudulent_Transactions  Account_Age  \
0  Mobile/Tablet   Nashville                                 2           25   
1         Tablet     Raleigh                                 3           18   
2  Mobile/Tablet  Charleston                                 4           85   
3         Tablet  Fort Worth                                 2           60   
4  Mobile/Tablet     Oakland                                 3           45   

   Number_of_Transactions_Last_24H Payment_Method  Fraudulent  
0                           

In [134]:
print(type(raw_cot))

<class 'str'>
